<a href="https://colab.research.google.com/github/tpedCode/P07/blob/feat%2Fmodeling/notebooks/1_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Implémentez un modèle de scoring**

### Objectif :
- prédire la probabilité de défaut d’un client
- optimiser un seuil métier (coût FN > FP)
- préparer un modèle pour une API

### **Initialisation**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# =========================================
# INSTALLATION DES DÉPENDANCES
# =========================================

!pip install mlflow -q
!pip install pyngrok -q

In [3]:
# =========================================
# IMPORTS
# =========================================

# -------- Data manipulation --------
import numpy as np
import pandas as pd

# -------- System --------
import gc
import os
import shutil

# -------- Visualization --------
import matplotlib.pyplot as plt

# -------- MLflow --------
import mlflow
import mlflow.sklearn

# -------- Machine Learning --------

# Model selection
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split,
)

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

# Metrics
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
)

In [4]:
# =========================================
# PATHS
# =========================================

BASE_PATH = "/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/data/"

# Chemins de données
RAW_PATH = os.path.join(BASE_PATH, "raw/")
PROCESSED_PATH = os.path.join(BASE_PATH, "processed/")

print(RAW_PATH)
print(PROCESSED_PATH)

/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/data/raw/
/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/data/processed/


### **Load data**

In [ ]:
# =========================================
# LOAD DATA
# =========================================

def load_csv(name):
    return pd.read_csv(os.path.join(RAW_PATH, name))

application_train = load_csv("application_train.csv")
application_test = load_csv("application_test.csv")

bureau = load_csv("bureau.csv")
bureau_balance = load_csv("bureau_balance.csv")

credit_card_balance = load_csv("credit_card_balance.csv")
installments_payments = load_csv("installments_payments.csv")
pos_cash_balance = load_csv("POS_CASH_balance.csv")
previous_application = load_csv("previous_application.csv")

sample_submission = load_csv("sample_submission.csv")

homecredit_columns_description = pd.read_csv(
    os.path.join(RAW_PATH, "HomeCredit_columns_description.csv"),
    encoding="latin1"
)

### **Fonctions génériques**

In [ ]:
def eda(
    df,
    name=None,

    # Blocs
    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    # Cat detail
    cat_detail_cols=None,

    # Graphs
    show_barplots=False,
    barplot_cols=None,

    show_boxplots=False,
    boxplot_cols=None,

    top_n=20
):

    if name is None:
        name_found = [k for k, v in globals().items() if v is df]
        name = name_found[0] if name_found else "DataFrame"

    print("\n" + "=" * 60)
    print(f"EDA : {name.upper()}")
    print("=" * 60)

    # ===== GENERAL =====
    if show_general:
        print("\n-----GENERAL------------------------------------------------")
        print(f"Shape : {df.shape}")
        print(f"Nb colonnes : {len(df.columns)}")

        print("\nColonnes :")
        print(df.columns.tolist())

        print("\nAperçu :")
        display(df.head())

    # ===== DATA QUALITY =====
    if show_quality:
        print("\n" + "-" * 60)
        print("DATA QUALITY")
        print("-" * 60)

        num_cols = df.select_dtypes(include=np.number).columns

        summary = pd.DataFrame(index=df.columns)
        summary["dtype"] = df.dtypes
        summary["missing_%"] = (df.isnull().mean() * 100).round(2)
        summary["n_unique"] = df.nunique()

        zero_pct = pd.Series(index=df.columns, dtype=float)
        zero_pct[num_cols] = (df[num_cols] == 0).mean() * 100
        summary["zero_%"] = zero_pct.fillna(0).round(2)

        summary["top_freq_%"] = (
            df.apply(lambda x: x.value_counts(normalize=True).iloc[0] * 100)
        ).round(2)

        display(summary.sort_values("missing_%", ascending=False))

    # ===== NUMERICAL =====
    if show_numerical:
        print("\n" + "-" * 60)
        print("NUMERICAL VARIABLES")
        print("-" * 60)

        num_cols = df.select_dtypes(include=np.number).columns
        display(df[num_cols].describe().T.round(2))

    # ===== CATEGORICAL =====
    if show_categorical:

        cat_cols = df.select_dtypes(include="object").columns

        if len(cat_cols) > 0:
            print("\n" + "-" * 60)
            print("CATEGORICAL VARIABLES")
            print("-" * 60)

            cat_summary = pd.DataFrame({
                "n_unique": df[cat_cols].nunique()
            }).sort_values(by="n_unique", ascending=False)

            display(cat_summary)

            if cat_detail_cols:
                print("\n--- Détail variables catégorielles ---")

                for col in cat_detail_cols:
                    if col in df.columns:
                        print(f"\n{col}")
                        display(
                            df[col]
                            .value_counts(normalize=True)
                            .round(2)
                            .head(top_n)
                        )

    # ===== BARPLOTS =====
    if show_barplots and barplot_cols:
        print("\n" + "-" * 60)
        print("BARPLOTS")
        print("-" * 60)

        for col in barplot_cols:
            if col in df.columns:
                plt.figure()

                counts_abs = df[col].value_counts().head(top_n)
                counts_pct = (counts_abs / len(df) * 100).round(2)

                ax = counts_pct.plot(kind="bar")

                plt.title(col)
                plt.ylabel("%")
                plt.xticks(rotation=90)

                for i, val in enumerate(counts_abs):
                    plt.text(
                        i,
                        counts_pct.iloc[i],
                        f"{val}",
                        ha="center",
                        va="bottom"
                    )

                plt.tight_layout()
                plt.show()

    # ===== BOXPLOTS =====
    if show_boxplots and boxplot_cols:
        print("\n" + "-" * 60)
        print("BOXPLOTS")
        print("-" * 60)

        for col in boxplot_cols:
            if col in df.columns and np.issubdtype(df[col].dtype, np.number):
                plt.figure()
                df.boxplot(column=col)
                plt.title(col)
                plt.tight_layout()
                plt.show()

In [ ]:
def get_doc(df_desc, column, table=None):
    """
    Affiche la description d'une colonne.

    Params:
    - df_desc : dataframe des descriptions
    - column : nom de la colonne
    - table : optionnel, nom de la table pour filtrer
    """

    result = df_desc[df_desc["Row"] == column]

    if table is not None:
        result = result[result["Table"] == table]

    print("\n" + "=" * 80)
    print(f"DOCUMENTATION : {column.upper()}")
    print("=" * 80)

    if result.empty:
        print("Aucune description trouvée")
        return

    for _, row in result.iterrows():
        print("\n" + "-" * 80)
        print(f"Table       : {row['Table']}")
        print(f"Colonne     : {row['Row']}")
        print("Description :")
        print(row["Description"])

# get_doc(homecredit_columns_description, "AMT_CREDIT")
# get_doc(homecredit_columns_description, "AMT_CREDIT", "application_train.csv")

In [ ]:
def one_hot_encoder(df):
    categorical_cols = df.select_dtypes(include="object").columns      # variables catégorielles
    df = pd.get_dummies(df, columns=categorical_cols, dummy_na=True)   # encodage
    return df

## **1. EXPLORATORY DATA ANALYSIS**

In [ ]:
eda(
    application_train,
    name="application_train",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=[
        "NAME_CONTRACT_TYPE",
        "CODE_GENDER",
        "NAME_INCOME_TYPE",
        "NAME_FAMILY_STATUS",
        "NAME_EDUCATION_TYPE",
        "ORGANIZATION_TYPE"
    ],

    show_barplots=True,
    barplot_cols=["TARGET"],

    show_boxplots=True,
    boxplot_cols=[
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]
)

In [ ]:
eda(
    application_test,
    name="application_test",

    show_general=True,
    show_quality=True,
    show_numerical=False,
    show_categorical=False
)

In [ ]:
eda(
    bureau_balance,
    name="bureau_balance",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=["STATUS"]
)

In [ ]:
eda(
    bureau,
    name="bureau",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=[
        "CREDIT_ACTIVE",
        "CREDIT_TYPE"
    ],

    show_barplots=False,
    show_boxplots=True,
    boxplot_cols=[
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT",
        "AMT_CREDIT_SUM_OVERDUE"
    ]
)

In [ ]:
eda(
    credit_card_balance,
    name="credit_card_balance",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=["NAME_CONTRACT_STATUS"],

    show_boxplots=True,
    boxplot_cols=[
        "AMT_BALANCE",
        "AMT_PAYMENT_CURRENT"
    ]
)

In [ ]:
eda(
    homecredit_columns_description,
    name="homecredit_columns_description",

    show_general=True,
    show_quality=np.format_float_scientific,
    show_numerical=False,
    show_categorical=False
)

get_doc(homecredit_columns_description, "AMT_CREDIT")

In [ ]:
eda(
    installments_payments,
    name="installments_payments",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=False,

    show_boxplots=True,
    boxplot_cols=[
        "AMT_PAYMENT",
        "AMT_INSTALMENT"
    ]
)

In [ ]:
eda(
    pos_cash_balance,
    name="pos_cash_balance",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=["NAME_CONTRACT_STATUS"]
)

In [ ]:
eda(
    previous_application,
    name="previous_application",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=[
        "NAME_CONTRACT_STATUS",
        "NAME_CLIENT_TYPE",
        "NAME_GOODS_CATEGORY"
    ],

    show_boxplots=True,
    boxplot_cols=[
        "AMT_APPLICATION",
        "AMT_CREDIT",
        "AMT_ANNUITY"
    ]
)

In [ ]:
eda(
    sample_submission,
    name="sample_submission",

    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=False
)

### **1. Synthèse globale**

#### **1.1. Structure du dataset**

* Dataset **relationnel multi-tables**, centré sur `application_train`
* Granularité différente selon les tables :
  * client (`application_train`)
  * crédit historique (`bureau`, `previous_application`)
  * transactions (`installments`, `credit_card`, `POS`)
* Clés principales :
  * `SK_ID_CURR` → client
  * `SK_ID_PREV` / `SK_ID_BUREAU` → crédits


#### **1.2. Qualité globale des données**

* **Beaucoup de missing values** sur certaines familles :
  * features logement (`_AVG`, `_MODE`, `_MEDI`) → \~70%
  * variables finance secondaires (bureau, previous)
* **Colonnes très déséquilibrées / constantes**
  * flags documents → quasi inutiles
  * certaines variables catégorielles dominées par une seule modalité
* **Outliers importants** sur montants (revenus, crédits)
* Variables fortement asymétriques


#### **1.3. Signaux métier importants**

* Target déséquilibrée (\~8%)
* Historique crédit présent sur plusieurs tables → **opportunité feature engineering**
* Variables externes (`EXT_SOURCE`) probablement très prédictives
* Données temporelles négatives (jours relatifs) → nécessite transformation


### **2. Synthèse par table**


#### **2.1. application\_train : table principale de modélisation**

**Points clés :**

* 307k lignes, 122 variables
* Target binaire déséquilibrée (\~8%)
* Profil clients :
  * majorité “Cash loans”
  * majorité revenus “Working”
  * majorité mariés
* Variables importantes :
  * `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`
  * `EXT_SOURCE_1/2/3`

**Problèmes :**

* Très forte proportion de missing sur variables logement (\~70%)
* Beaucoup de variables inutiles :
  * `FLAG_DOCUMENT_X`
* variance faible sur certaines colonnes binaires

**Opportunités :**

* forte richesse de variables catégorielles
* bonnes candidates pour encoding
* création de ratios :
  * crédit / revenu
  * annuité / revenu


#### **2.2. application\_test : équivalent du train sans target**

**Points clés :**

* structure identique
* même problématique de missing

**Usage :**

* transformation identique au train


#### **2.3 bureau**

**Rôle : historique crédit externe (crédit bureau)**

**Points clés :**

* 1.7M lignes
* 63% crédits CLOSED, 37% ACTIVE
* types dominants :
  * Consumer credit (73%)
  * Credit card (23%)

**Problèmes :**

* missing élevés :
  * `AMT_ANNUITY`, `AMT_CREDIT_MAX_OVERDUE`
* variables peu informatives (overdue très souvent 0)

**Opportunités :**

* agrégation par client :
  * nombre de crédits actifs
  * dette totale
  * ratio dette / crédit
  * historique de défaut


#### **2.4. bureau\_balance : historique mensuel des crédits bureau**

**Points clés :**

* table volumineuse (\~27M lignes)
* `STATUS` très informatif :
  * C (closed) \~50%
  * 0 (no delay) \~27%
  * X (unknown) \~21%

**Opportunités :**

* feature temporelle :
  * nombre de mois en défaut
  * worst status
* enrichissement bureau


#### **2.5. previous\_application : historique des demandes de crédit internes**

**Points clés :**

* 1.6M lignes
* statuts :
  * Approved (\~62%)
  * Canceled (\~19%)
  * Refused (\~17%)
* clients :
  * repeater majoritaire (\~74%)

**Problèmes :**

* énormément de missing (jusqu’à 99% sur certaines variables)
* variables financières partielles

**Opportunités :**

* comportements passés :
  * taux de refus
  * nb d'applications
  * profil client historique


#### **2.6. credit\_card\_balance : historique cartes de crédit**

**Points clés :**

* 3.8M lignes
* 96% des contrats “Active”

**Problèmes :**

* beaucoup de variables à 0 dominant
* missing modéré (\~20% sur certaines variables)

**Opportunités :**

* indicateurs comportement :
  * utilisation du crédit
  * paiements
  * dépassements


#### **2.7. installments\_payments : historique des remboursements**

**Points clés :**

* 13.6M lignes
* données complètes (quasi pas de missing)

**Opportunités majeures :**

* calcul du retard :
  * `DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT`
* indicateurs :
  * retards moyens
  * fréquence des retards
  * écart paiement vs dû


#### **2.8. POS\_CASH\_balance : crédits POS (point of sale)**

**Points clés :**

* 10M lignes
* majorité ACTIVE (\~91%)

**Problèmes :**

* variables fortement dominées (DPD = 0)

**Opportunités :**

* comportement crédit court terme
* évolution remboursements


#### **2.9. sample\_submission : template de soumission**

**Aucun intérêt analytique**


#### **2.10. homecredit\_columns\_description : dictionnaire de données**

**Usage :**

* compréhension des variables uniquement


### **3. Conclusion opérationnelle**

#### **3.1. Points forts du dataset**

* richesse des sources (multi-tables)
* historique complet client
* données comportementales détaillées


#### **3.2. Points faibles**

* beaucoup de missing structurés
* nombreuses variables inutiles
* nécessité d’agrégations complexes


#### **3.3. Directions ML prioritaires**

* Nettoyage :
  * suppression variables très manquantes / constantes
* Feature engineering :
  * agrégations multi-tables
  * variables retard paiement
  * ratios financiers
* Encodage catégoriel
* gestion du déséquilibre target

## **2. SETUP MLFLOW**

### **2.1. Configuration**

In [5]:
# Reset

# Réinitialisation complète de MLflow.
# Supprime :
# - la base SQLite (mlflow.db)
# - l'ensemble des runs
# - les artefacts enregistrés
#
# À utiliser lorsqu'on souhaite repartir d'un environnement MLflow propre.

MLFLOW_PATH = "/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/mlflow"

if os.path.exists(MLFLOW_PATH):
    shutil.rmtree(MLFLOW_PATH)

print("MLflow reset")

MLflow reset


In [6]:
# Nom de l'expérience MLflow
# Toutes les exécutions (runs) du projet seront regroupées dans cette expérience
EXPERIMENT_NAME = "home_credit_scoring_v1"

# Racine du projet sur Google Drive
# Permet de centraliser toutes les ressources du projet (data, MLflow, notebooks, etc.)
PROJECT_PATH = "/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07"

# Dossier dédié à MLflow
# Contiendra :
# - la base SQLite (mlflow.db)
# - les artefacts (modèles sauvegardés, fichiers générés)
MLFLOW_PATH = os.path.join(PROJECT_PATH, "mlflow")

# Dossier de stockage des artefacts MLflow
# Les modèles entraînés seront enregistrés ici
ARTIFACTS_PATH = os.path.join(MLFLOW_PATH, "artifacts")

# Création automatique des dossiers si inexistants
os.makedirs(ARTIFACTS_PATH, exist_ok=True)

# Configuration du backend MLflow
# La base SQLite est stockée sur Drive pour conserver l'historique même après fermeture de la session Colab
mlflow.set_tracking_uri(
    f"sqlite:///{MLFLOW_PATH}/mlflow.db"
)

# Création de l'expérience si elle n'existe pas déjà
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_PATH}"
    )

# Sélection de l'expérience active
# Tous les futurs runs seront enregistrés dans cette expérience
mlflow.set_experiment(EXPERIMENT_NAME)

# Récupération des informations de l'expérience
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Flag pratique pour activer/désactiver MLflow rapidement sans modifier le reste du notebook
RUN_MLFLOW = True

2026/07/02 13:08:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/02 13:08:31 INFO mlflow.store.db.utils: Updating database tables


In [7]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Artifact location :", experiment.artifact_location)
print("Experiment ID :", experiment.experiment_id)

Tracking URI : sqlite:////content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/mlflow/mlflow.db
Artifact location : file:///content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/mlflow/artifacts
Experiment ID : 1


### **2.2. Test**

In [ ]:
# =========================================
# TEST MLFLOW
# =========================================

# Copie du dataset principal
# Permet de faire un test sans modifier les données d'origine
baseline_data = application_train.copy()

# Sélection de quelques variables simples
# Objectif : valider MLflow rapidement sans lancer le pipeline complet
X_test_mlflow = baseline_data[
    ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]
].fillna(0)

# Variable cible
y_test_mlflow = baseline_data["TARGET"]

# Séparation train / validation
# La stratification conserve la proportion de défauts (~8%)
X_train, X_val, y_train, y_val = train_test_split(
    X_test_mlflow,
    y_test_mlflow,
    stratify=y_test_mlflow,
    test_size=0.2,
    random_state=42
)

# Modèle simple pour le test technique
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

MODEL_NAME = model.__class__.__name__

# Entraînement
model.fit(X_train, y_train)

# Prédiction des probabilités
y_proba = model.predict_proba(X_val)[:, 1]

# Calcul de l'AUC
auc = roc_auc_score(y_val, y_proba)

print(f"AUC test MLflow : {auc:.4f}")

# =========================================
# LOGGING MLFLOW
# =========================================

FEATURE_VERSION = "test_mlflow"

if RUN_MLFLOW:

    with mlflow.start_run(
        run_name=f"{MODEL_NAME}_{FEATURE_VERSION}"
    ):

        # Paramètres
        mlflow.log_param("model", MODEL_NAME)
        mlflow.log_param("feature_version", FEATURE_VERSION)
        mlflow.log_param("n_features", X_test_mlflow.shape[1])

        # Métriques
        mlflow.log_metric("auc", auc)

        # Sauvegarde du modèle
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model"
        )

print("Test MLflow terminé")

## **3. FEATURE ENGINEERING**

### **3.1. Calculs et transformations**

In [ ]:
# =========================================
# BASE APPLICATION
# =========================================

# Copie du dataset principal
feature_engineering = application_train.copy()

# Correction anomalie DAYS_EMPLOYED (valeur aberrante connue)
feature_engineering["DAYS_EMPLOYED_ANOM"] = (
    feature_engineering["DAYS_EMPLOYED"] == 365243                                    # Flag anomalie
)
feature_engineering["DAYS_EMPLOYED"] = (
    feature_engineering["DAYS_EMPLOYED"].replace(365243, np.nan)                      # Remplacement par NA
)

# Variables temporelles (plus interprétables)
feature_engineering["AGE"] = -feature_engineering["DAYS_BIRTH"]                       # Âge en jours (positif)
feature_engineering["YEARS_EMPLOYED"] = (
    -feature_engineering["DAYS_EMPLOYED"] / 365                                       # Ancienneté en années
)

# Ratios financiers (très importants pour le risque)
feature_engineering["CREDIT_INCOME_RATIO"] = (
    feature_engineering["AMT_CREDIT"] / feature_engineering["AMT_INCOME_TOTAL"]       # Niveau d’endettement
)
feature_engineering["ANNUITY_INCOME_RATIO"] = (
    feature_engineering["AMT_ANNUITY"] / feature_engineering["AMT_INCOME_TOTAL"]      # Charge mensuelle
)
feature_engineering["PAYMENT_RATE"] = (
    feature_engineering["AMT_ANNUITY"] / feature_engineering["AMT_CREDIT"]            # Proxy durée crédit
)
feature_engineering["INCOME_PER_PERSON"] = (
    feature_engineering["AMT_INCOME_TOTAL"] / feature_engineering["CNT_FAM_MEMBERS"]  # Revenu par personne
)

# Moyenne des scores externes → très prédictif
feature_engineering["EXT_SOURCE_MEAN"] = feature_engineering[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
].mean(axis=1)

# Encodage des variables catégorielles
feature_engineering = one_hot_encoder(feature_engineering)

print("Application:", feature_engineering.shape)


# =========================================
# BUREAU (historique crédit externe)
# =========================================

bureau_df = one_hot_encoder(bureau.copy())                                            # Encodage

# Agrégations par client
bureau_agg = bureau_df.groupby("SK_ID_CURR").agg({
    "AMT_CREDIT_SUM": ["mean", "sum"],                                                # Montant crédit
    "AMT_CREDIT_SUM_DEBT": ["mean", "sum"],                                           # Dette
    "DAYS_CREDIT": ["min", "mean"],                                                   # Historique crédit
    "CREDIT_DAY_OVERDUE": ["max", "mean"]                                             # Retards
})

# Renommage des colonnes
bureau_agg.columns = ["BURO_" + "_".join(col).upper() for col in bureau_agg.columns]

# Jointure avec dataset principal
feature_engineering = feature_engineering.join(
    bureau_agg, how="left", on="SK_ID_CURR"
)

# Nettoyage mémoire
del bureau_df, bureau_agg
gc.collect()


# =========================================
# PREVIOUS APPLICATION (historique demandes)
# =========================================

prev = one_hot_encoder(previous_application.copy())                                   # Encodage

# Ratio demandé / obtenu
prev["APP_CREDIT_PERC"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"]

# Agrégations
prev_agg = prev.groupby("SK_ID_CURR").agg({
    "AMT_APPLICATION": ["mean"],
    "AMT_CREDIT": ["mean"],
    "APP_CREDIT_PERC": ["mean"],
    "CNT_PAYMENT": ["mean"]
})

# Renommage
prev_agg.columns = ["PREV_" + "_".join(col).upper() for col in prev_agg.columns]

# Jointure
feature_engineering = feature_engineering.join(
    prev_agg, how="left", on="SK_ID_CURR"
)

# Nettoyage mémoire
del prev, prev_agg
gc.collect()


# =========================================
# INSTALLMENTS (comportement paiement)
# =========================================

ins = installments_payments.copy()

# % payé vs dû
ins["PAYMENT_PERC"] = ins["AMT_PAYMENT"] / ins["AMT_INSTALMENT"]

# Retard de paiement
ins["DPD"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]
ins["DPD"] = ins["DPD"].apply(lambda x: x if x > 0 else 0)                            # On garde seulement les retards

# Agrégations
ins_agg = ins.groupby("SK_ID_CURR").agg({
    "PAYMENT_PERC": ["mean"],                                                         # Qualité de paiement
    "DPD": ["max", "mean", "sum"]                                                     # Retards
})

# Renommage
ins_agg.columns = ["INSTAL_" + "_".join(col).upper() for col in ins_agg.columns]

# Jointure
feature_engineering = feature_engineering.join(
    ins_agg, how="left", on="SK_ID_CURR"
)

# Nettoyage mémoire
del ins, ins_agg
gc.collect()

eda(
    feature_engineering,
    name="feature_engineering",
    show_general=True,
    show_quality=True,
    show_numerical=True,
    show_categorical=True,

    cat_detail_cols=[
        "NAME_INCOME_TYPE",
        "NAME_FAMILY_STATUS",
        "CODE_GENDER"
    ],

    show_barplots=True,
    barplot_cols=[
        "TARGET"
    ],

    show_boxplots=True,
    boxplot_cols=[
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "PAYMENT_RATE",
        "EXT_SOURCE_MEAN",
        "BURO_AMT_CREDIT_SUM_MEAN",
        "BURO_AMT_CREDIT_SUM_DEBT_SUM",
        "INSTAL_DPD_MEAN",
        "INSTAL_DPD_MAX"
    ]
)

#### **Structure générale du dataset**

Le dataset final contient 307 511 observations pour 286 variables. Il est construit autour de la table principale `application_train`, enrichie avec des variables issues du feature engineering et d’agrégations multi-tables.

On observe trois grandes familles de variables :

* variables brutes initiales (revenu, crédit, données démographiques)
* variables dérivées (ratios, transformations temporelles)
* variables agrégées issues des historiques (`BURO_*`, `PREV_*`, `INSTAL_*`)
* variables encodées (one-hot encoding des catégories)

#### **Feature engineering réalisé**

Le feature engineering enrichit fortement l’information initiale grâce à :

* transformation des variables temporelles  
  (âge, ancienneté en années)

* traitement d’anomalies métier  
  (`DAYS_EMPLOYED = 365243` remplacé et transformé en indicateur)

* création de ratios financiers très informatifs :
  * `CREDIT_INCOME_RATIO`
  * `ANNUITY_INCOME_RATIO`
  * `PAYMENT_RATE`
  * `INCOME_PER_PERSON`

* combinaison des scores externes :
  * `EXT_SOURCE_MEAN` (fort signal prédictif)

* intégration de signaux comportementaux via agrégations :
  * historique bureau (`BURO_*`)
  * historique des demandes (`PREV_*`)
  * comportement de paiement (`INSTAL_*`)

#### **Qualité des données**

Plusieurs points ressortent :

* un taux élevé de valeurs manquantes persiste pour certaines variables logement (\~60–70%)
* présence de nombreuses variables binaires issues du one-hot encoding
* certaines colonnes sont fortement déséquilibrées ou quasi constantes (flags, catégories rares)
* introduction de valeurs extrêmes ou infinies (ex : `INSTAL_PAYMENT_PERC_MEAN = inf`)

Ces éléments indiquent la nécessité :

* d’un nettoyage ou filtrage de certaines features
* d’un contrôle des valeurs aberrantes

#### **ariables numériques**

Les variables numériques montrent :

* forte dispersion sur les montants (revenus, crédits)
* distributions asymétriques (beaucoup de valeurs faibles et quelques extrêmes)
* variables comportementales intéressantes :
  * `INSTAL_DPD_*` (retards de paiement)
  * ratios financiers (endettement, capacité de remboursement)

Ces variables sont généralement les plus prédictives dans les modèles de scoring.

### **Variables catégorielles**
#
Après encodage :

* explosion du nombre de colonnes (one-hot encoding)
* transformation des variables catégorielles en variables binaires
* disparition des colonnes d’origine (difficile d’analyse directe)

Certaines catégories restent peu représentées et peuvent introduire du bruit.

#### **Variables issues des historiques**

Les features multi-tables apportent un gain important :

* `BURO_*` : situation d’endettement externe
* `PREV_*` : comportement passé vis-à-vis du crédit
* `INSTAL_*` : discipline de paiement

Ces variables capturent le risque client de manière dynamique et sont essentielles pour la performance du modèle.

#### **Points de vigilance**

* présence de valeurs infinies (`inf`) à traiter avant modélisation
* nombreuses features peu informatives (one-hot rares)
* variables très corrélées (notamment ratios et montants)
* possible surapprentissage dû au grand nombre de variables

#### **Conclusion**

Le dataset final est riche et combine :

* informations socio-économiques
* ratios financiers pertinents
* historique comportemental client

Le feature engineering améliore significativement la qualité du signal, notamment grâce aux données historiques.

Les prochaines étapes doivent se concentrer sur :

* nettoyage des valeurs extrêmes
* réduction de dimension (feature selection)
* comparaison de modèles pour exploiter efficacement cette richesse de variables

In [ ]:
# SAVE
feature_engineering.to_parquet(
    os.path.join(PROCESSED_PATH, "feature_engineering.parquet")
)

In [ ]:
# # LOAD
# feature_engineering = pd.read_parquet(
#     os.path.join(PROCESSED_PATH, "feature_engineering.parquet")
# )

### **3.2. Préparation pour modélisations futures**

In [ ]:
# Séparation features / target
X_raw = feature_engineering.drop(columns=["TARGET", "SK_ID_CURR"])  # features uniquement
y = feature_engineering["TARGET"]                                   # target

print("X_raw shape :", X_raw.shape)
print("y shape :", y.shape)

In [ ]:
# Copie pour garder version brute
X_clean = X_raw.copy()

# Gestion des valeurs infinies : emplacement des inf par NaN
X_clean = X_clean.replace([np.inf, -np.inf], np.nan)

# Gestion des valeurs manquantes : emplacement simple des NA (baseline)
X_clean = X_clean.fillna(0)

# Suppression des colonnes constantes
constant_cols = X_clean.nunique() <= 1
print("Colonnes constantes supprimées :", constant_cols.sum())

X_clean = X_clean.loc[:, ~constant_cols]

# Contrôle final
print("X_clean shape :", X_clean.shape)
print("Nb NaN restants :", X_clean.isna().sum().sum())

In [ ]:
# Réduction corrélation forte

corr_matrix = X_clean.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.99)]

print("Colonnes supprimées (corrélation forte) :", len(to_drop))

X_clean = X_clean.drop(columns=to_drop)

print("X_clean final :", X_clean.shape)

In [ ]:
# SAVE
X_clean.to_parquet(
    os.path.join(PROCESSED_PATH, "X_clean.parquet")
)

y.to_frame().to_parquet(
    os.path.join(PROCESSED_PATH, "y.parquet")
)

### **3.3. LOAD**

In [8]:
# LOAD
X_clean = pd.read_parquet(
    os.path.join(PROCESSED_PATH, "X_clean.parquet")
)
y = pd.read_parquet(
    os.path.join(PROCESSED_PATH, "y.parquet")
)

## **4. MODELING**

### **4.1. Fonctions génériques**

#### **4.1.1. Metrics business**

In [9]:
def compute_business_cost(y_true, y_pred, fn_weight=10, fp_weight=1):
    """
    Calcule le coût métier basé sur :
    - FP : False Positive : bon client prédit mauvais
    - FN : False Negative : mauvais client prédit bon client

    Paramètres :
    - y_true : valeurs réelles (0/1)
    - y_pred : prédictions binaires (0/1)
    - fn_weight : poids des faux négatifs (défaut non détecté)
    - fp_weight : poids des faux positifs (bon client refusé)

    Retour :
    - business_cost (float) :
        coût total pondéré des erreurs de classification.
        Calcul = (fn_weight × nombre de FN) + (fp_weight × nombre de FP)

        → Plus la valeur est élevée, plus le modèle est mauvais du point de vue métier.
        → La valeur est non normalisée (dépend du nombre d'observations)
    """

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel() # Matrice de confusion pour récupérer facilement FP et FN
    business_cost = fn_weight * fn + fp_weight * fp           # Coût métier : FN pénalisé fortement (perte financière réelle) / FP pénalisé faiblement (manque à gagner)
    return business_cost

In [10]:
def optimize_threshold(y_true, y_proba, fn_weight=10, fp_weight=1, n_thresholds=100):
    """
    Trouve le seuil optimal minimisant le coût métier.

    Paramètres :
    - y_true : valeurs réelles
    - y_proba : probabilités du modèle
    - fn_weight : coût FN (important métier)
    - fp_weight : coût FP
    - n_thresholds : nombre de seuils testés (précision vs temps)


    Retour :
    - best_threshold (float) :
        seuil optimal pour le modèle, minimisant le coût métier

    - best_cost (float) :
        coût métier minimum atteint par le modèle
        (correspond au best_threshold)

    - results_df (DataFrame) :
        tableau contenant :
        - tous les seuils testés
        - le coût métier associé à chaque seuil
        → permet d’analyser la sensibilité du coût au seuil
        → utile pour visualisation (courbe seuil vs coût)
    """

    # On teste plusieurs seuils entre 0 et 1
    # → permet de remplacer le seuil par défaut (0.5)
    thresholds = np.linspace(0, 1, n_thresholds)

    results = []

    for threshold in thresholds:

        # Transformation des probabilités en classes
        # → dépend du seuil testé
        y_pred = (y_proba > threshold).astype(int)

        # Calcul du coût métier pour ce seuil
        cost = compute_business_cost(
            y_true, y_pred,
            fn_weight=fn_weight,
            fp_weight=fp_weight
        )

        # Stockage pour analyse et comparaison
        results.append({
            "threshold": threshold,
            "business_cost": cost
        })

    # Conversion en DataFrame → utile pour analyse graphique
    results_df = pd.DataFrame(results)

    # Sélection du seuil minimisant le coût métier
    best_row = results_df.loc[results_df["business_cost"].idxmin()]

    best_threshold = best_row["threshold"]
    best_cost = best_row["business_cost"]

    return best_threshold, best_cost, results_df

#### **4.1.2. Pipeline d’entraînement**

In [11]:
def split_data(X, y, test_size=0.2, random_state=42):
    """
    Sépare les données en train / validation.

    Objectif :
    - entraîner sur une partie des données
    - évaluer sur une autre (généralisation du modèle)

    Choix techniques :
    - stratify=y : conserve la proportion de la classe minoritaire (~8%)
    - test_size=0.2 : compromis classique (80% train / 20% validation)
    - random_state : reproductibilité

    Choix métier :
    - dataset déséquilibré → il est crucial de garder la même distribution

    Retour :
    - X_train : DataFrames des variables explicatives pour l'entraînement
    - X_val : DataFrames des variables explicatives pour la validation
    - y_train : Series de la variable cible pour l'entraînement
    - y_val : Series de la variable cible pour la validation

    → Les données sont séparées de façon aléatoire mais stratifiée,
      ce qui garantit que la proportion des classes (ex : défaut / non défaut)
      est similaire entre train et validation.

    → Les tailles dépendent de test_size :
      - train : (1 - test_size)
      - validation : test_size
    """

    return train_test_split(
        X,                        # données d'entrée (features)
        y,                        # variable cible à prédire
        stratify=y,               # conserve la proportion des classes
        test_size=test_size,      # taille du jeu de validation (ex : 20%)
        random_state=random_state # garantit un split reproductible
    )

In [12]:
def scale_data(X_train, X_val):
    """
    Standardise les variables (centrage + réduction).

    Objectif :
    - rendre les variables comparables (même échelle)
    - améliorer la stabilité et la performance des modèles sensibles aux échelles

    Choix techniques :
    - StandardScaler → transforme chaque variable : moyenne = 0, écart-type = 1
    - fit uniquement sur X_train → calcule moyenne et écart-type sur le train seulement
    - transform sur X_val → applique les mêmes paramètres sans recalcul (évite le data leakage)
    - reconstruction en DataFrame → conservation des noms de colonnes et des index

    Choix métier :
    - aucun direct → transformation purement technique
    - impact indirect : améliore la qualité du modèle donc la décision métier

    Retour :
    - X_train_scaled : DataFrame des variables d'entraînement standardisées avec noms de colonnes conservés
    - X_val_scaled : DataFrame des variables de validation standardisées avec les mêmes paramètres que le train
    """

    scaler = StandardScaler()                       # initialise l'objet de normalisation (apprendra moyenne et écart-type)
    X_train_scaled = scaler.fit_transform(X_train)  # apprend les statistiques sur X_train puis applique la transformation
    X_val_scaled = scaler.transform(X_val)          # applique la transformation sur X_val avec les mêmes paramètres (pas de fit)

    X_train_scaled = pd.DataFrame(                  # reconstruit un DataFrame pour garder noms de colonnes et index
        X_train_scaled,
        columns=X_train.columns,
        index=X_train.index
    )

    X_val_scaled = pd.DataFrame(                    # même reconstruction pour cohérence avec le train
        X_val_scaled,
        columns=X_val.columns,
        index=X_val.index
    )

    return X_train_scaled, X_val_scaled

In [13]:
def train_model_cv(model, param_grid, X_train, y_train):
    """
    Entraîne un modèle avec validation croisée et sélection automatique
    des meilleurs hyperparamètres.

    Objectif :
    - apprendre plusieurs versions du modèle avec différentes configurations
    - identifier celle qui généralise le mieux
    - retourner un modèle déjà entraîné et optimisé

    Choix techniques :
    - StratifiedKFold → conserve la proportion de défauts dans chaque fold
    - validation croisée à 5 folds → évaluation plus robuste qu'un simple split
    - GridSearchCV → teste toutes les combinaisons d'hyperparamètres
    - scoring="roc_auc" → comparaison basée sur la capacité à classer les clients
      du plus risqué au moins risqué
    - n_jobs=-1 → parallélisation des calculs

    Choix métier :
    - l'objectif ici est uniquement de trouver le meilleur modèle technique
    - le coût métier n'est pas encore pris en compte car il dépend du seuil
    - l'optimisation métier sera réalisée après obtention des probabilités

    Retour :
    - best_model : meilleur modèle entraîné sur l'ensemble du train
    - best_params : hyperparamètres ayant obtenu la meilleure AUC
    """
    cv = StratifiedKFold(
        n_splits=5,             # découpe les données en 5 folds pour la validation croisée
        shuffle=True,           # mélange les données avant découpage pour éviter un biais d'ordre
        random_state=42         # garantit la reproductibilité des splits
    )

    grid = GridSearchCV(
        estimator=model,        # modèle à entraîner et optimiser
        param_grid=param_grid,  # grille des hyperparamètres à tester
        scoring="roc_auc",      # métrique utilisée pour comparer les modèles (indépendante du seuil)
        cv=cv,                  # stratégie de validation croisée définie ci-dessus
        n_jobs=-1,              # utilise tous les cœurs CPU pour accélérer le calcul
        verbose=1               # affiche la progression de la recherche
    )

    grid.fit(X_train, y_train)  # entraîne tous les modèles (toutes combinaisons d’hyperparamètres) avec validation croisée

    return grid.best_estimator_, grid.best_params_  # renvoie le meilleur modèle déjà entraîné trouvé et ses hyperparamètres associés

In [14]:
def predict_proba(model, X_val):
    """
    Génère les probabilités de défaut pour chaque client.

    Objectif :
    - estimer le risque de défaut client
    - produire un score exploitable pour l'AUC et les décisions métier

    Choix techniques :
    - predict_proba retourne une probabilité pour chaque classe :
        [P(classe 0), P(classe 1)]
    - seule la probabilité de la classe positive (TARGET=1) est conservée
    - aucune décision n'est encore prise à ce stade

    Choix métier :
    - la probabilité permet d'évaluer finement le risque client
    - la décision d'accepter ou refuser sera prise plus tard via un seuil

    Retour :
    - tableau de probabilités de défaut compris entre 0 et 1
    - une valeur par client :
        0   → risque très faible
        1   → risque très élevé
    """

    # Garde uniquement P(TARGET = 1) = probabilité de défaut
    return model.predict_proba(X_val)[:, 1]

In [15]:
def evaluate_auc(y_val, y_proba):
    """
    Calcule l'AUC (Area Under the ROC Curve).

    Objectif :
    - mesurer la capacité du modèle à classer les clients du plus risqué au moins risqué
    - évaluer la qualité des scores de risque produits par le modèle

    Choix techniques :
    - compare les probabilités prédites (y_proba) aux vraies classes (y_val)
    - utilise les probabilités directement, sans transformer en 0/1
    - indépendant de tout seuil de décision
    - adapté aux jeux de données déséquilibrés

    Interprétation :
    L'AUC ne vérifie pas si les probabilités sont exactes, elle vérifie si les clients réellement en défaut reçoivent
    généralement des probabilités plus élevées que les autres, elle mesure donc la qualité du classement des risques

    Exemple :
    y_val   = [0, 1, 0, 1]
    y_proba = [0.10, 0.80, 0.20, 0.90]

    Les clients en défaut reçoivent les scores les plus élevés : classement parfait → AUC = 1.0

    Choix métier :
    - utile pour comparer plusieurs modèles de manière neutre
    - ne représente pas directement le coût financier réel
    - ne permet pas à lui seul de prendre une décision d'acceptation/refus

    Retour :
    - score AUC compris entre 0 et 1

        1.0  → séparation parfaite
        0.5  → modèle aléatoire
        <0.5 → classement inversé

    - plus l'AUC est élevée, plus le modèle distingue correctement les clients risqués des clients non risqués
    """

    return roc_auc_score(y_val, y_proba)    # Compare les probabilités prédites aux vraies classes et mesure la qualité du classement des clients par risque.

In [16]:
def log_mlflow(model, best_params, auc, cost, threshold, X, scale, model_name):
    """
    Enregistre une expérience de machine learning dans MLflow.

    Objectif :
    - conserver un historique complet des expérimentations
    - comparer facilement plusieurs modèles et configurations
    - assurer la traçabilité des résultats
    - sauvegarder le modèle entraîné pour une réutilisation future (batch, API, monitoring)

    Choix techniques :
    - création d'un run MLflow identifié par le nom du modèle et la version des features
    - journalisation des paramètres d'entraînement :
        * type de modèle
        * hyperparamètres optimaux
        * nombre de features
        * utilisation du scaling
    - journalisation des métriques :
        * AUC
        * coût métier
        * seuil optimal
    - sauvegarde du modèle entraîné comme artefact MLflow
    - enregistrement automatique dans le Model Registry

    Choix métier :
    - permet de comparer les performances techniques et les performances métier
    - garantit la reproductibilité des résultats
    - facilite le déploiement du meilleur modèle
    - permet de savoir précisément quelle version du modèle a été utilisée en production

    Retour :
    - aucun retour (None)
    - les informations sont enregistrées dans le serveur ou le stockage MLflow configuré
    """

    if RUN_MLFLOW:

        RUN_VERSION = "v1"               # Version du modèle entraîné
        FEATURE_VERSION = "features_v1"  # Version du feature engineering

        # Démarre un nouvel enregistrement MLflow
        with mlflow.start_run(
            run_name=f"{model_name}_{FEATURE_VERSION}_{RUN_VERSION}"
        ):

            # Paramètres utilisés pour l'entraînement
            mlflow.log_param("model", model.__class__.__name__)
            mlflow.log_param("best_params", str(best_params))
            mlflow.log_param("n_features", X.shape[1])
            mlflow.log_param("scale", scale)
            mlflow.log_param("feature_version", FEATURE_VERSION)

            # Résultats obtenus par le modèle
            mlflow.log_metric("auc", auc)
            mlflow.log_metric("business_cost", cost)
            mlflow.log_metric("threshold", threshold)

            # Sauvegarde du modèle entraîné
            # comme artefact ET dans le Model Registry
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
                registered_model_name=model_name
            )

            print(
                f"Modèle '{model_name}' enregistré dans le Model Registry."
            )

In [17]:
def run_model(
    model,
    param_grid,
    X,
    y,
    model_name="model",
    scale=False
):
    """
    Exécute le pipeline complet d'entraînement et d'évaluation d'un modèle.

    Objectif :
    - entraîner un modèle de scoring de bout en bout
    - comparer les performances techniques et métier
    - produire un modèle prêt à être enregistré et réutilisé

    Choix techniques :
    - séparation train / validation pour évaluer la généralisation
    - scaling optionnel selon les besoins du modèle
    - validation croisée + recherche d'hyperparamètres
    - évaluation par AUC à partir des probabilités prédites
    - optimisation du seuil pour minimiser le coût métier

    Choix métier :
    - séparation entre qualité du modèle (AUC) et qualité de la décision (coût métier)
    - recherche du seuil le plus rentable
    - traçabilité complète via MLflow

    Retour :
    - results : dictionnaire standardisé contenant :
        * modèle
        * AUC
        * coût métier
        * seuil optimal
        * hyperparamètres optimaux

    - optimized_model : meilleur modèle entraîné
    """

    # 1. Split train / validation
    X_train, X_val, y_train, y_val = split_data(X, y)

    # 2. Scaling optionnel
    if scale:
        X_train, X_val = scale_data(X_train, X_val)

    # 3. Entraînement + optimisation des hyperparamètres
    optimized_model, best_params = train_model_cv(
        model,
        param_grid,
        X_train,
        y_train
    )

    # 4. Estimation du risque pour chaque observation
    y_proba = predict_proba(
        optimized_model,
        X_val
    )

    # 5. Qualité du classement des observations
    auc = evaluate_auc(
        y_val,
        y_proba
    )

    # 6. Recherche du seuil minimisant le coût métier
    threshold, cost, _ = optimize_threshold(
        y_val,
        y_proba
    )

    # 7. Sauvegarde de l'expérience dans MLflow
    log_mlflow(
        optimized_model,
        best_params,
        auc,
        cost,
        threshold,
        X,
        scale,
        model_name
    )

    # 8. Résumé standardisé des résultats
    results = {
        "model": model_name,
        "auc": auc,
        "business_cost": cost,
        "threshold": threshold,
        "best_params": best_params
    }

    print(results)

    return results, optimized_model

#### **4.1.3. Analyses**

In [18]:
def global_feature_importance(model, X, top_n=20):
    """
    Analyse l'importance globale des variables.

    Objectif :
    - identifier les variables les plus influentes dans le modèle
    - comprendre les drivers principaux du risque

    Choix techniques :
    - utilisation de feature_importances_ (modèles arbres)
    - tri décroissant pour visualisation

    Limite :
    - ne donne pas le sens (positif/négatif)
    - dépend du type de modèle

    Choix métier :
    - permet d’identifier les facteurs de risque principaux
    - utile pour interprétation globale et présentation
    """

    # Récupération des importances
    importances = model.feature_importances_

    # Création DataFrame
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": importances
    })

    # Tri décroissant
    fi_df = fi_df.sort_values("importance", ascending=False)

    # Affichage top variables
    print(fi_df.head(top_n))

    # Visualisation
    fi_df.head(top_n).plot.barh(
        x="feature",
        y="importance",
        figsize=(8, 6)
    )

    return fi_df

In [19]:
def local_feature_analysis(model, X, y_proba, n_random=2):
    """
    Analyse de cas individuels.

    Objectif :
    - comprendre pourquoi un client est classé à risque ou non
    - apporter une explicabilité locale

    Cas analysés :
    - probabilité max (client très risqué)
    - probabilité min (client très sûr)
    - probabilité médiane
    - cas aléatoires

    Choix technique :
    - sélection basée sur y_proba
    - approche simple sans SHAP (plus rapide)

    Choix métier :
    - permet d’expliquer des décisions individuelles
    - utile pour API et confiance utilisateur
    """

    df = pd.DataFrame({
        "proba": y_proba
    })

    # Sélection des cas intéressants
    idx_max = df["proba"].idxmax()
    idx_min = df["proba"].idxmin()
    idx_med = (df["proba"] - df["proba"].median()).abs().idxmin()

    idx_random = np.random.choice(df.index, n_random, replace=False)

    selected_idx = [idx_max, idx_min, idx_med] + list(idx_random)

    print("Index sélectionnés :", selected_idx)

    # Affichage des features associées
    for idx in selected_idx:
        print("\n--- Client index :", idx)
        print("Probabilité :", y_proba[idx])
        print(X.iloc[idx])

In [20]:
def global_model_analysis(y_val, y_proba, threshold):
    """
    Analyse globale du modèle.

    Objectif :
    - comprendre la distribution des scores
    - analyser les erreurs (FN / FP)
    - relier performance technique et impact métier

    Choix techniques :
    - histogramme des probabilités
    - confusion matrix basée sur threshold optimisé

    Choix métier :
    - FN = erreurs critiques (défaut non détecté)
    - FP = erreurs acceptables (client refusé)

    → permet de valider l’adéquation du modèle au besoin business
    """

    # ----------------------------
    # 1. Distribution des scores
    # ----------------------------
    plt.hist(y_proba, bins=50)
    plt.title("Distribution des probabilités")
    plt.xlabel("Probabilité de défaut")
    plt.ylabel("Fréquence")
    plt.show()

    # ----------------------------
    # 2. Prédictions avec seuil
    # ----------------------------
    y_pred = (y_proba > threshold).astype(int)

    # ----------------------------
    # 3. Matrice de confusion
    # ----------------------------
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    print("Confusion matrix :")
    print("TN :", tn)
    print("FP :", fp)
    print("FN :", fn)
    print("TP :", tp)

    # ----------------------------
    # 4. Interprétation métier
    # ----------------------------
    print("\nAnalyse métier :")
    print(f"Faux négatifs (FN) : {fn} → clients risqués non détectés (coût élevé)")
    print(f"Faux positifs (FP) : {fp} → clients refusés à tort (coût modéré)")

    print(f"\nSeuil utilisé : {threshold}")

### **4.2. Modèles**

#### **4.1.1. Baseline : DummyClassifier**

In [21]:
# Hyperparamètres testés
param_grid_dummy = {
    # "prior" : prédit systématiquement selon la proportion observée des classes dans le jeu d'entraînement.
    # Exemple : TARGET = 8% le modèle produira toujours une probabilité ≈ 0.08
    # Sert uniquement de référence minimale.
    "strategy": ["prior"]
}

results_dummy, model_dummy = run_model(
    model=DummyClassifier(
        random_state=42
    ),
    param_grid=param_grid_dummy,
    X=X_clean,
    y=y,
    model_name="DummyClassifier",
    scale=False
)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Modèle 'DummyClassifier' enregistré dans le Model Registry.
{'model': 'DummyClassifier', 'auc': np.float64(0.5), 'business_cost': np.float64(49650.0), 'threshold': np.float64(0.08080808080808081), 'best_params': {'strategy': 'prior'}}


Successfully registered model 'DummyClassifier'.
Created version '1' of model 'DummyClassifier'.


#### **4.1.2. Modèle simple : LogisticRegression**

In [22]:
param_grid_logreg = {
    # Intensité de la régularisation.
    # Petit C : modèle plus simple, moins de risque de surapprentissage
    # Grand C : modèle plus flexible risque plus élevé d'overfitting
    # Exploration sur plusieurs ordres de grandeur, pratique classique pour les modèles linéaires.
    "C": [0.01, 0.1, 1, 10],

    # Gestion du déséquilibre de classes, TARGET ≈ 8%.
    # None : poids identiques pour toutes les classes.
    # balanced : sklearn augmente automatiquement le poids de la classe minoritaire.
    # Très pertinent dans un contexte de scoring crédit.
    "class_weight": [None, "balanced"]
}

results_logreg, model_logreg = run_model(
    model=LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    param_grid=param_grid_logreg,
    X=X_clean,
    y=y,
    model_name="LogisticRegression",
    scale=True
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Modèle 'LogisticRegression' enregistré dans le Model Registry.
{'model': 'LogisticRegression', 'auc': np.float64(0.7520566637943192), 'business_cost': np.float64(33341.0), 'threshold': np.float64(0.5151515151515152), 'best_params': {'C': 0.1, 'class_weight': 'balanced'}}


Successfully registered model 'LogisticRegression'.
Created version '1' of model 'LogisticRegression'.


#### **4.1.3. Modèle complexe : LGBMClassifier**

In [23]:
param_grid_lgbm = {
    # Nombre d'arbres.
    # Plus il y a d'arbres, plus le modèle peut capturer des relations complexes.
    # Valeurs raisonnables pour une première recherche.
    "n_estimators": [100, 300],

    # Vitesse d'apprentissage.
    # Petite valeur : apprentissage plus lent mais souvent meilleure généralisation.
    # Valeurs classiques pour LightGBM.
    "learning_rate": [0.01, 0.05, 0.1],

    # Profondeur maximale des arbres.
    # Contrôle la complexité du modèle.
    # Plus la profondeur est grande, plus le risque de surapprentissage augmente.
    "max_depth": [3, 5, 7],

    # Gestion du déséquilibre.
    # Même logique que pour LogisticRegression.
    "class_weight": [None, "balanced"]
}

results_lgbm, model_lgbm = run_model(
    model=LGBMClassifier(
        random_state=42,
        verbosity=-1
    ),
    param_grid=param_grid_lgbm,
    X=X_clean,
    y=y,
    model_name="LGBMClassifier",
    scale=False
)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


ValueError: 
All the 180 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1049, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/engine.py", line 297, in train
    booster = Booster(params=params, train_set=train_set)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/pyth

### **4.3. Analyses**

#### **4.3.1. Feature importance globale**

#### **4.3.2. Feature importance locale**

#### **4.3.3. Synthèse générale**